In [1]:
import sys
!{sys.executable} -m pip install requests pandas psycopg

Der Befehl "c:\Users\user\OneDrive" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


In [1]:
import requests
import datetime
import pandas as pd
import urllib3
import psycopg
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:

base_url = "https://redispatch-run.azurewebsites.net/api/operations/get"
params_template = {
    "networkoperator": "edi",
    "type": "finished",
    "orderDirection": "desc",
    "orderBy": "start",
    "chunkNr": 1,
    "param1": "start",
    "op1": "equals",
    "startOp": "eq"
}

results = []
start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2025, 12, 31)

for day in range((end_date - start_date).days + 1):
    date_str = (start_date + datetime.timedelta(days=day)).isoformat()
    chunk_nr = 1

    while True:
        params = params_template.copy()
        params["val1"] = date_str
        params["chunkNr"] = chunk_nr
        response = requests.get(base_url, params=params, verify=False)

        if response.status_code == 200:
            data = response.json()
            if "operations" in data and data["operations"] and len(data["operations"]) > 0:
                for operation in data["operations"]:
                    results.append(operation)
                    # Zeige nur die Anzahl der Datenpunkte an
                    print(f"Datenpunkte hinzugefügt: {len(results)}", end="\r")
                chunk_nr += 1
            else:
                # Keine weiteren Daten für diesen Tag
                break
        else:
            # Fehler oder keine Daten
            break

In [3]:
len(results)

609093

In [4]:

# Datenbankverbindung konfigurieren mit psycopg
db_config = {
    "dbname": "redispatch_losses",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost",
    "port": 5432,
    "sslmode": "require"
}

# In DataFrame umwandeln
df = pd.DataFrame(results)

# Mapping der API-Spaltennamen auf Datenbank-Spaltennamen
column_mapping = {
    'start': 'star',
    'end': 'ende',
    'operationId': 'operationid',
    'srId': 'srid',
    'liabilityOfCompensation': 'liabilityofcompensation',
    'DSOKey': 'dsokey',
    'inducingNO': 'inducingno',
    'KalKID': 'kalkid',
    'controlStage': 'controlstage',
    'locationBottleneck': 'locationbottleneck',
    'assetKey': 'assetkey',
    'Reason': 'reasontwo'  # Großes R ist reasontwo, kleines r bleibt reason
}

# Spaltennamen umbenennen falls vorhanden
df = df.rename(columns=column_mapping)

# Spaltennamen anpassen, falls nötig (eventnummer NICHT im DataFrame, wird von DB vergeben)
expected_columns = ['id','operationid','srid','star','ende','duration','location','locationbottleneck','controlstage','reason','reasontwo','assetkey','requestor','liabilityofcompensation','dsokey','inducingno','kalkid']
df = df.reindex(columns=expected_columns)

# Konvertiere Datentypen für die Datenbank
if not df.empty:
    if 'star' in df.columns:
        df['star'] = pd.to_datetime(df['star'], errors='coerce')
    if 'ende' in df.columns:
        df['ende'] = pd.to_datetime(df['ende'], errors='coerce')
    if 'duration' in df.columns:
        df['duration'] = pd.to_numeric(df['duration'], errors='coerce')

    # Speichere datetime-Spalten vor der where-Operation
    star_values = df['star'].copy() if 'star' in df.columns and df['star'].notna().any() else None
    ende_values = df['ende'].copy() if 'ende' in df.columns and df['ende'].notna().any() else None

    # Ersetze NaN mit None für alle Spalten
    df = df.where(pd.notnull(df), None)

    # Konvertiere datetime64 zu Python datetime
    if star_values is not None:
        df['star'] = star_values.apply(lambda x: x.to_pydatetime() if pd.notnull(x) else None)
    if ende_values is not None:
        df['ende'] = ende_values.apply(lambda x: x.to_pydatetime() if pd.notnull(x) else None)

    # Mit psycopg in die Datenbank schreiben
    try:
        conn = psycopg.connect(**db_config)
        cursor = conn.cursor()

        # INSERT Statement vorbereiten (verlust bleibt NULL)
        insert_query = """
        INSERT INTO redispatch.raw_data_edis (
            id, operationid, srid, star, ende, duration,
            location, locationbottleneck, controlstage,
            reason, reasontwo, assetkey, requestor,
            liabilityofcompensation, dsokey, inducingno, kalkid,
            verlust
        ) VALUES (
            %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, NULL
        )
        """

        # Daten als Liste von Tupeln vorbereiten - verwende values.tolist() statt to_records()
        data_tuples = [tuple(row) for row in df.values.tolist()]

        # Batch-Insert durchführen
        cursor.executemany(insert_query, data_tuples)

        # Commit
        conn.commit()

        print(f"\n✓ {len(data_tuples)} Datensätze erfolgreich in redispatch.raw_data_edis importiert!")

        # Anzahl der Datensätze in der Zieltabelle prüfen
        cursor.execute("SELECT COUNT(*) FROM redispatch.raw_data_edis")
        count = cursor.fetchone()[0]
        print(f"✓ Gesamtanzahl Datensätze  {count}")

        cursor.close()
        conn.close()

    except Exception as e:
        print(f"\n❌ Fehler beim Import in Datenbank: {e}")
        import traceback
        traceback.print_exc()
        if 'conn' in locals():
            conn.rollback()
            conn.close()
else:
    print("Keine Daten zum Schreiben in die Datenbank.")


✓ 609093 Datensätze erfolgreich in redispatch.raw_data_edis importiert!
✓ Gesamtanzahl Datensätze  609093
